# E-Commerce Conversion Prediction
## Summer Analytics 2026 - Week 2 Hackathon

**Objective:** Predict whether a user converts (makes a purchase) based on demographics, browsing behavior, and session activity.

**Evaluation Metric:** F1 Score

## 1. Import Libraries

In [ ]:
import sys
import os

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_data, explore_data, plot_feature_importance, plot_threshold_optimization, create_submission
from src.feature_engineering import create_features
from src.preprocessing import prepare_data
from src.model import EnsembleModel, optimize_threshold, evaluate_model

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

## 2. Load Data

In [ ]:
# Load datasets
train_df, public_test_df, private_test_df = load_data(
    train_path='../data/train.csv',
    public_test_path='../data/public_test.csv',
    private_test_path='../data/private_test.csv'
)

print("\nData loaded successfully!")

## 3. Exploratory Data Analysis

In [ ]:
# Explore training data
explore_data(train_df, target_col='Converted')

In [ ]:
# Display first few rows
print("\nFirst 5 rows of training data:")
train_df.head()

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
train_df['Converted'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Target Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Converted')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Not Converted (0)', 'Converted (1)'], rotation=0)

# Percentage plot
train_df['Converted'].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Target Distribution (Percentage)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Converted')
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(['Not Converted (0)', 'Converted (1)'], rotation=0)

plt.tight_layout()
plt.savefig('../outputs/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClass Imbalance Ratio: {(train_df['Converted']==0).sum() / (train_df['Converted']==1).sum():.2f}:1")

In [ ]:
# Correlation analysis
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
print("\nTop 10 correlations with 'Converted':")
target_corr = corr_matrix['Converted'].sort_values(ascending=False)
print(target_corr[1:11])

## 4. Feature Engineering

In [ ]:
# Apply feature engineering to all datasets
train_enhanced = create_features(train_df.copy())
public_test_enhanced = create_features(public_test_df.copy())
private_test_enhanced = create_features(private_test_df.copy())

print(f"Original features: {train_df.shape[1]}")
print(f"Enhanced features: {train_enhanced.shape[1]}")
print(f"\nNew features added: {train_enhanced.shape[1] - train_df.shape[1]}")

print("\nNew feature columns:")
new_cols = [col for col in train_enhanced.columns if col not in train_df.columns]
print(new_cols)

## 5. Data Preprocessing

In [ ]:
# Prepare training data (train + public_test for more robust preprocessing)
combined_train = pd.concat([train_enhanced, public_test_enhanced], axis=0, ignore_index=True)

# Store User_ID before preprocessing
train_user_ids = train_enhanced['User_ID']
public_user_ids = public_test_enhanced['User_ID']
private_user_ids = private_test_enhanced['User_ID']

# Drop User_ID from datasets
train_enhanced = train_enhanced.drop('User_ID', axis=1)
public_test_enhanced = public_test_enhanced.drop('User_ID', axis=1)
private_test_enhanced = private_test_enhanced.drop('User_ID', axis=1)

# Prepare data with preprocessing
X_train, y_train, X_public_test, preprocessor = prepare_data(
    train_enhanced, 
    public_test_enhanced
)

# Also prepare private test
private_test_processed = private_test_enhanced.drop('Converted', axis=1, errors='ignore')
X_private_test = preprocessor.transform(private_test_processed)

# Get public test labels
y_public_test = public_test_enhanced['Converted']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_public_test shape: {X_public_test.shape}")
print(f"y_public_test shape: {y_public_test.shape}")
print(f"X_private_test shape: {X_private_test.shape}")

## 6. Model Training

In [ ]:
# Calculate class imbalance ratio
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Initialize and train ensemble model
model = EnsembleModel(random_state=42)
model.fit(X_train, y_train, scale_pos_weight=scale_pos_weight)

print("\nModel training complete!")

## 7. Threshold Optimization

In [ ]:
# Predict probabilities on public test set
public_proba = model.predict_proba(X_public_test)[:, 1]

# Optimize threshold
best_threshold, best_f1, threshold_results = optimize_threshold(
    y_public_test, 
    public_proba, 
    start=0.20, 
    end=0.80, 
    step=0.01
)

print(f"\nOptimal Threshold: {best_threshold:.2f}")
print(f"Best F1 Score: {best_f1:.4f}")

# Plot threshold optimization
plot_threshold_optimization(threshold_results, best_threshold, best_f1)
plt.savefig('../outputs/threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation on Public Test Set

In [ ]:
# Predict with optimal threshold
public_predictions = model.predict(X_public_test, threshold=best_threshold)

# Evaluate
print("\n" + "="*60)
print("PUBLIC TEST SET EVALUATION")
print("="*60)
public_metrics = evaluate_model(y_public_test, public_predictions, public_proba)

## 9. Feature Importance Analysis

In [ ]:
# Get feature importance
feature_names = X_train.columns.tolist()
importance_df = model.get_feature_importance(feature_names, top_n=20)

print("\nTop 20 Most Important Features:")
print(importance_df[['feature', 'importance']].to_string(index=False))

# Plot feature importance
plot_feature_importance(importance_df, top_n=20, figsize=(10, 8))
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Retrain on Full Labeled Data

In [ ]:
# Combine train and public_test for final training
X_full = pd.concat([X_train, X_public_test], axis=0, ignore_index=True)
y_full = pd.concat([y_train, y_public_test], axis=0, ignore_index=True)

print(f"Full training set size: {X_full.shape}")

# Retrain model
final_model = EnsembleModel(random_state=42)
final_model.fit(X_full, y_full, scale_pos_weight=scale_pos_weight)

print("\nFinal model trained on full labeled data!")

## 11. Generate Predictions for Private Test Set

In [ ]:
# Predict on private test set
private_predictions = final_model.predict(X_private_test, threshold=best_threshold)

print(f"Private test predictions generated: {len(private_predictions)}")
print(f"\nPrediction distribution:")
print(pd.Series(private_predictions).value_counts())

## 12. Create Submission File

In [ ]:
# Create submission
submission_df = create_submission(
    user_ids=private_user_ids,
    predictions=private_predictions,
    output_path='../outputs/submission.csv'
)

print("\nSubmission file preview:")
print(submission_df.head(10))

## 13. Summary of Results

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"\nModel: XGBoost + LightGBM Ensemble")
print(f"Public Test F1 Score: {best_f1:.4f}")
print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"\nTop 5 Features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")
print(f"\nSubmission file: outputs/submission.csv")
print(f"Total predictions: {len(private_predictions)}")
print("\n" + "="*60)